# Milestone - Unit 2: the instruments

The starter steers at a point three waypoints ahead and never looks further.
Unit 2 gave the car instruments: where it is relative to the track, and what is coming.
`agents/unit-02.py` is the starter's driving, unchanged, plus those instruments,
measured every tick and stored on the agent (`cross_track_error`, `segment_index`, `curvature_ahead`).

This notebook asserts each gate of the unit, using the submission's own functions:

- **G2.1** the geometry functions match analytic circles: Menger curvature, the cross-track error on a polyline circle in both directions, `wrap_angle`'s edge cases, and the heading of a chord; and Menger matches the track generator's curvature;
- **G2.2** the cross-track error, computed from telemetry with the pinned rules, equals racekit's `offset` on every row of the deterministic `monza-like` race; the in-car instrument agrees to `float32` sensor precision;
- **G2.3** the submission races exactly like the starter and passes `check_submission`;
- **G2.4** the `STEER_GAIN` sweep on `test-circuit`: the car runs wider as the gain falls, and at gain 1 it crashes;
- **G2.5** the cross-track scorecard Units 4 and 5 are scored against.

## 1. The submission and its functions

The milestone runs from `projects/racekit/milestones/`, so the submission is one directory up.
`load_submission` imports it under the shim; its module holds the geometry functions.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.check_submission import check_submission
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track

AGENT = Path("..") / "agents" / "unit-02.py"
TOL = 1e-9  # race times are sums of 0.05 s ticks, so "exact" means to float rounding

with shim_installed():
    Agent, module_name = load_submission(AGENT)
unit02 = sys.modules[module_name]
wrap_angle = unit02.wrap_angle
project_onto_segment = unit02.project_onto_segment
menger_curvature = unit02.menger_curvature
print("geometry functions:", wrap_angle.__name__, unit02.to_car_frame.__name__, project_onto_segment.__name__, menger_curvature.__name__)


def cross_track(px, py, xs, ys, hint, reach=8):
    """The pinned rules: segments hint - reach .. hint + reach (cyclic), the smallest clamped-foot
    distance wins (strict <, ties to the first), the offset is the signed distance to that segment's line."""
    n = len(xs)
    best_d2, best, best_offset = math.inf, hint % n, 0.0
    for k in range(hint - reach, hint + reach + 1):
        i = k % n
        j = (i + 1) % n
        _, d2, offset = project_onto_segment(px, py, xs[i], ys[i], xs[j], ys[j])
        if d2 < best_d2:
            best_d2, best, best_offset = d2, i, offset
    return best, best_offset


def track_position(telemetry, xs, ys):
    """cross_track over every row, the hint carried from 0 and reset to 0 on a respawn-hold row."""
    hint, segments, offsets = 0, [], []
    for x, y, held in zip(telemetry["x"], telemetry["y"], telemetry["held"]):
        hint, offset = cross_track(float(x), float(y), xs, ys, 0 if held else hint)
        segments += [hint]
        offsets += [offset]
    return np.array(segments), np.array(offsets)


def waypoints(track):
    return [float(value) for value in track.x], [float(value) for value in track.y]

## 2. G2.1: the functions against analytic circles

- Three points on a circle of radius R have Menger curvature 1/R to 1e-9, positive counter-clockwise (a left turn) and negative clockwise.
- On a **counter-clockwise** polyline circle (left is the inside), a point at radius r has cross-track error R − r, within the chord's sagitta R(1 − cos(Δθ/2)); driven clockwise, the sign flips.
- `wrap_angle` maps ±π and ±3π to −π and 0 to 0, stays in [−π, π), and is idempotent.
- The `atan2` heading of a chord equals the tangent's heading at the chord's **midpoint** angle, to 1e-12.

In [ ]:
worst_kappa = 0.0
for R in [18.0, 25.0, 47.0, 120.0, 1000.0]:
    for start, step in [(0.0, 2.0 / R), (0.7, 0.3), (-2.9, 1.3)]:
        p = [(5.0 + R * math.cos(start + k * step), -8.0 + R * math.sin(start + k * step)) for k in range(3)]
        worst_kappa = max(worst_kappa,
                          abs(menger_curvature(*p[0], *p[1], *p[2]) - 1.0 / R),
                          abs(menger_curvature(*p[2], *p[1], *p[0]) + 1.0 / R))
assert worst_kappa < 1e-9

R, N = 50.0, 60
angles = np.linspace(0.0, 2.0 * math.pi, N + 1)[:-1]
ccw_x, ccw_y = [R * math.cos(a) for a in angles], [R * math.sin(a) for a in angles]
cw_x, cw_y = ccw_x[::-1], ccw_y[::-1]
sagitta = R * (1.0 - math.cos(math.pi / N))
worst_ccw = worst_cw = 0.0
for r in [43.0, 48.5, 50.0, 52.0, 57.0]:
    for phi in np.linspace(0.02, 6.25, 41):
        px, py = r * math.cos(phi), r * math.sin(phi)
        start = int(np.argmin(np.hypot(np.array(ccw_x) - px, np.array(ccw_y) - py)))
        worst_ccw = max(worst_ccw, abs(cross_track(px, py, ccw_x, ccw_y, start)[1] - (R - r)))
        worst_cw = max(worst_cw, abs(cross_track(px, py, cw_x, cw_y, N - 1 - start)[1] + (R - r)))
assert worst_ccw <= sagitta and worst_cw <= sagitta

for angle in [math.pi, -math.pi, 3 * math.pi, -3 * math.pi]:
    assert wrap_angle(angle) == -math.pi
assert wrap_angle(0.0) == 0.0
for angle in np.linspace(-25.0, 25.0, 501):
    once = wrap_angle(angle)
    assert -math.pi <= once < math.pi and wrap_angle(once) == once
    assert abs(math.sin(once) - math.sin(angle)) < 1e-12 and abs(math.cos(once) - math.cos(angle)) < 1e-12

worst_chord = 0.0
for theta1, theta2 in [(0.0, 0.11), (1.0, 1.4), (3.0, 3.3), (-2.2, -1.0), (5.5, 5.6)]:
    for R in [18.0, 120.0]:
        chord = math.atan2(R * (math.sin(theta2) - math.sin(theta1)), R * (math.cos(theta2) - math.cos(theta1)))
        worst_chord = max(worst_chord, abs(wrap_angle(chord - ((theta1 + theta2) / 2.0 + math.pi / 2.0))))
assert worst_chord < 1e-12

print(f"Menger vs 1/R: {worst_kappa:.1e} | cross-track vs R - r: ccw {worst_ccw:.4f} m, cw {worst_cw:.4f} m (sagitta {sagitta:.4f} m) | chord heading {worst_chord:.1e} rad")

### Menger against the track generator

The generator stores each track's curvature; the Menger curvature of each waypoint and its two neighbours must match it within 1e-3 1/m,
with left turns positive, and the tightest radius must be 120 / 18 / 25 m.

In [ ]:
R_MIN = {"oval": 120.00, "test-circuit": 18.00, "monza-like": 25.00}
tracks, kappas = {}, {}
for name, r_min in R_MIN.items():
    track = load_track(name)
    xs, ys = waypoints(track)
    n = len(xs)
    kappa = np.array([menger_curvature(xs[i - 1], ys[i - 1], xs[i], ys[i], xs[(i + 1) % n], ys[(i + 1) % n]) for i in range(n)])
    tracks[name], kappas[name] = track, kappa
    gap = float(np.max(np.abs(kappa - track.curvature)))
    measured_r_min = 1.0 / float(np.max(np.abs(kappa)))
    print(f"{name:>12}: max |Menger - generator| {gap:.1e} 1/m, R_min {measured_r_min:.2f} m")
    assert gap < 1e-3 and abs(measured_r_min - r_min) < 0.005
    curved = np.abs(track.curvature) > 1e-3
    assert np.sum(np.sign(kappa[curved]) != np.sign(track.curvature[curved])) == 0  # left turns positive, like the generator

## 3. G2.3: the submission races exactly like the starter

At jitter 0 the starter's times are 223.50 s (`oval`), 412.75 s (`test-circuit`) and 882.55 s (`monza-like`), all clean.
The instruments are measured every tick but never touch the controls, so the submission's times must be the same, to the tick.

In [ ]:
STARTER_TIMES = {"oval": 223.50, "test-circuit": 412.75, "monza-like": 882.55}
races = {}
for name, pinned in STARTER_TIMES.items():
    races[name] = run_race(AGENT, name, car=KinematicCar(jitter=0.0))
    race = races[name]
    print(race)
    assert race.finished and race.collisions == 0
    assert abs(race.total_time - pinned) <= TOL

report = check_submission(AGENT)
print(report)
assert report.ok, report.findings

## 4. G2.2: instrument equivalence

The cross-track error from the telemetry's `x, y`, with the pinned rules, must equal racekit's `offset` within 1e-6 m on **every** row of the deterministic `monza-like` race.

Then the in-car instrument: a recording subclass reads `cross_track_error` after each step.
Step k sees the car where tick k − 1 left it, so it is compared with row k − 1.
The sensors are `float32`, as on the real interface, so the agreement is to about 1e-5 m, not 1e-13.

In [ ]:
monza_tel = races["monza-like"].telemetry
mx, my = waypoints(tracks["monza-like"])
monza_segments, monza_offsets = track_position(monza_tel, mx, my)
worst = float(np.max(np.abs(monza_offsets - monza_tel["offset"])))
print(f"monza-like: {len(monza_tel)} rows, largest |cross-track - offset| {worst:.1e} m")
assert len(monza_tel) == 17651 and worst < 1e-6

in_car = {}


class Recorder(Agent):
    async def initialize(self):
        await super().initialize()
        in_car["cross_track"], in_car["curvature_ahead"] = [], []

    async def step(self):
        await super().step()
        in_car["cross_track"] += [self.cross_track_error]
        in_car["curvature_ahead"] += [self.curvature_ahead]


recorded = run_race(Recorder, "test-circuit", car=KinematicCar(jitter=0.0))
log = np.array(in_car["cross_track"])
in_car_gap = float(np.max(np.abs(log[1:] - recorded.telemetry["offset"][:-1])))
sharpest = float(np.max(np.abs(in_car["curvature_ahead"])))
print(f"test-circuit in-car: largest gap to the offset {in_car_gap:.1e} m; sharpest curvature ahead {sharpest:.4f} 1/m (R {1.0 / sharpest:.1f} m)")
assert abs(recorded.total_time - STARTER_TIMES["test-circuit"]) <= TOL
assert in_car_gap < 1e-4
assert abs(1.0 / sharpest - 18.0) < 0.01  # it sees the 18 m hairpin coming

## 5. G2.4: the `STEER_GAIN` sweep

`test-circuit`, 20 m/s, jitter 0, `max_seconds=1500`.
Times are exact; max |offset| (on the driving rows) is within ±0.005 m of the table; gains 8, 4 and 2 finish clean;
gain 1 does not finish and crashes exactly 16 times.
Gain 8 is the submission itself, already raced in §3.

In [ ]:
SWEEP = {  # gain: (finished, collisions, total_time, max |offset|)
    8.0: (True, 0, 412.75, 0.51),
    4.0: (True, 0, 413.55, 1.76),
    2.0: (True, 0, 415.35, 4.40),
    1.0: (False, 16, 1500.00, 5.64),
}
tc_kappa = kappas["test-circuit"]
tc_x, tc_y = waypoints(tracks["test-circuit"])
with TemporaryDirectory() as scratch:
    for gain, (finished, collisions, total_time, max_offset) in SWEEP.items():
        if gain == 8.0:
            race = races["test-circuit"]
        else:
            variant = write_variant(AGENT, Path(scratch) / f"gain_{gain:g}.py", {"STEER_GAIN": str(gain)})
            race = run_race(variant, "test-circuit", car=KinematicCar(jitter=0.0), max_seconds=1500)
        driving = race.telemetry["held"] == 0
        offsets = race.telemetry["offset"][driving]
        peak = float(np.max(np.abs(offsets)))
        print(f"gain {gain:3.0f}: finished {race.finished}, {race.collisions:2d} collisions, {race.total_time:7.2f} s, max |offset| {peak:.3f} m")
        assert race.finished == finished and race.collisions == collisions
        assert abs(race.total_time - total_time) <= TOL
        assert abs(peak - max_offset) <= 0.005
        if gain == 8.0:  # in the corners the car runs wide: offset and curvature have opposite signs
            segments, _ = track_position(race.telemetry, tc_x, tc_y)
            kappa = tc_kappa[segments[driving]]
            corner = np.abs(kappa) > 0.01
            assert np.sum(offsets[corner] * kappa[corner] >= 0) == 0

## 6. G2.5: the cross-track scorecard

The numbers Units 4 and 5 are scored against, at `TARGET_SPEED` 20, on the driving rows (`held == 0`).

- Jitter 0: max / RMS |offset| of 0.07 / 0.050 (`oval`), 0.51 / 0.088 (`test-circuit`) and 0.37 / 0.056 m (`monza-like`).
- Jitter 0.05, seeds 0–4: the median and the range (max − min) of the five max |offset| values and of the five RMS values,
  to ±0.001 m for the max and ±0.0005 m for the RMS.

The RMS is the steadier statistic (a range of 0.004–0.010 m against 0.11–0.16 m for the max), so later units are scored on it.

In [ ]:
DETERMINISTIC = {"oval": (0.07, 0.050), "test-circuit": (0.51, 0.088), "monza-like": (0.37, 0.056)}
JITTERED = {  # track: (max median, max range, RMS median, RMS range)
    "oval": (0.499, 0.164, 0.1397, 0.0099),
    "test-circuit": (0.689, 0.132, 0.1586, 0.0055),
    "monza-like": (0.611, 0.106, 0.1416, 0.0040),
}


def max_and_rms(race):
    offsets = race.telemetry["offset"][race.telemetry["held"] == 0]
    return float(np.max(np.abs(offsets))), float(np.sqrt(np.mean(offsets ** 2)))


print("      track | jitter 0: max  RMS | jitter 0.05, seeds 0-4: max median (range)  RMS median (range)")
for name in ["oval", "test-circuit", "monza-like"]:
    peak, rms = max_and_rms(races[name])
    assert abs(peak - DETERMINISTIC[name][0]) < 0.005 and abs(rms - DETERMINISTIC[name][1]) < 0.0005
    peaks, rmss = [], []
    for seed in range(5):
        race = run_race(AGENT, name, car=KinematicCar(jitter=0.05), seed=seed)
        assert race.finished and race.collisions == 0
        seed_peak, seed_rms = max_and_rms(race)
        peaks += [seed_peak]
        rmss += [seed_rms]
    stats = (float(np.median(peaks)), max(peaks) - min(peaks), float(np.median(rmss)), max(rmss) - min(rmss))
    print(f"{name:>11} | {peak:13.2f} {rms:6.3f} | {stats[0]:25.3f} ({stats[1]:.3f}) {stats[2]:12.4f} ({stats[3]:.4f})")
    for value, pinned, tolerance in zip(stats, JITTERED[name], (0.001, 0.001, 0.0005, 0.0005)):
        assert abs(value - pinned) <= tolerance, (name, stats)

## The instruments, pinned

| | `oval` | `test-circuit` | `monza-like` |
|---|---|---|---|
| R_min (m) | 120 | 18 | 25 |
| starter, jitter 0: max / RMS \|offset\| (m) | 0.07 / 0.050 | 0.51 / 0.088 | 0.37 / 0.056 |
| starter, jitter 0.05: RMS \|offset\| median (range) (m) | 0.1397 (0.0099) | 0.1586 (0.0055) | 0.1416 (0.0040) |

Jitter shows here too: a noisy car sits further off the line (an RMS of about 0.14–0.16 m against 0.05–0.09 m).
A later unit's controller earns a claim of "follows the line better" only by beating the RMS median by more than the larger range, at 0 collisions.